# 09:46 路径分歧与同状态局部资金价值：必要秩检验

固定负向假说；Qlib来源与原生Tiered局部标签均为封存对象，保留完整503交易日与不可行动/未成熟掩码。0拟合、0账户。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from scipy.stats import spearmanr

root=Path.cwd();card=root/"research/stage1_0946_pair_path_joint_value_rank_20261024"
plan_path=card/"plan.json";plan=json.loads(plan_path.read_text());src=plan["sources"]
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and plan["budget"]["new_market_GETs"]==plan["budget"]["model_fits"]==plan["budget"]["native_accounts"]==0
feature=root/src["feature_card"];labels_card=root/src["label_card"]
for path,key in [(feature/"plan.json","feature_plan_sha256"),(feature/"result.json","feature_result_sha256"),(feature/"summary.json","feature_summary_sha256"),(labels_card/"summary.json","label_summary_sha256"),(root/src["matched_control_plan"],"matched_control_plan_sha256")]:
 assert sha(path)==src[key],key
feature_result=json.loads((feature/"result.json").read_text())
assert feature_result["status"]=="source_and_novelty_preflight_pass" and all(feature_result["gates"].values())
assert feature_result["historical_first_public_qualified"] is False
lab_summary=json.loads((labels_card/"summary.json").read_text())
assert lab_summary["nas_evidence"]["sha256"]==src["label_NAS_sha256"] and lab_summary["nas_evidence"]["size_bytes"]==src["label_NAS_size_bytes"]
assert lab_summary["nas_readback"]["verified"] and lab_summary["nas_readback"]["scoped_restored_labels_sha256"]==src["label_artifact_sha256"]
artifact=labels_card/"qlib_recorder"/src["label_experiment_id"]/src["label_recorder_id"]/"artifacts/analysis/labels"
assert sha(artifact)==src["label_artifact_sha256"]
assert sha(root/src["minute_provider"]/"qlib_store_manifest.json")==src["minute_provider_manifest_sha256"]
assert sha(root/src["day_provider"]/"qlib_store_manifest.json")==src["day_provider_manifest_sha256"]
qlib.init(provider_uri=str((root/src["day_provider"]).resolve()),region="us",expression_cache=None,dataset_cache=None,kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time="2021-01-01",end_time="2022-12-30",freq="day"))
assert len(calendar)==503
with R.uri_context(str(labels_card/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["label_experiment_id"],recorder_id=src["label_recorder_id"])
 labels=rec.load_object("analysis/labels")
labels=labels.copy();labels["trade_date"]=pd.to_datetime(labels.trade_date).dt.normalize()
assert labels.trade_date.is_unique and len(labels)==1061
labels=labels.set_index("trade_date").reindex(calendar)
available=pd.to_datetime(labels.label_available_at)
cutoff=pd.Timestamp("2023-01-01T00:00:00-05:00").tz_localize(None)
eligible=labels.eligible.fillna(False).astype(bool)&labels.joint_local_fee_net_advantage.notna()&(available<cutoff)
assert int(eligible.sum())==329
qlib.init(provider_uri=str((root/src["minute_provider"]).resolve()),region="us",expression_cache=None,dataset_cache=None,kernels=1)
bars=D.features(["SOXL","SOXX"],["$open","$close","$price_valid"],start_time="2021-01-04 09:31:00",end_time="2022-12-30 16:00:00",freq="1min").reset_index()
assert {"instrument","datetime","$open","$close","$price_valid"}.issubset(bars.columns)
bars["datetime"]=pd.to_datetime(bars.datetime)
opening=bars.loc[bars.datetime.dt.strftime("%H:%M").between("09:31","09:45")].copy()
opening["day"]=opening.datetime.dt.normalize()
values={}
for day,g in opening.groupby("day",sort=True):
 expected=pd.date_range(day+pd.Timedelta(hours=9,minutes=31),periods=15,freq="min")
 parts={sym:g.loc[g.instrument.eq(sym)].sort_values("datetime") for sym in ("SOXL","SOXX")}
 good=True
 for x in parts.values():
  good &= len(x)==15 and pd.DatetimeIndex(x.datetime).equals(expected)
  if len(x)==15:good &= bool(np.isfinite(x[["$open","$close","$price_valid"]].to_numpy(float)).all() and (x["$open"]>0).all() and (x["$close"]>0).all() and x["$price_valid"].eq(1).all())
 if not good:continue
 sx,xx=parts["SOXL"],parts["SOXX"]
 rl=np.log(sx["$close"].to_numpy(float)/sx["$open"].to_numpy(float))
 rx=np.log(xx["$close"].to_numpy(float)/xx["$open"].to_numpy(float))
 values[day]=float(np.square(rl-3*rx).sum()/(np.square(rl).sum()+9*np.square(rx).sum()+1e-12))
coordinate=pd.Series(values,dtype=float).reindex(calendar)
assert int(coordinate.notna().sum())==503 and coordinate.between(0,2.01).all()
mask=eligible&coordinate.notna()
y=labels.loc[mask,"joint_local_fee_net_advantage"].to_numpy(float)
x=coordinate.loc[mask].to_numpy(float)
assert len(x)==len(y)==329 and np.isfinite(x).all() and np.isfinite(y).all()
full=float(spearmanr(x,y).statistic)
annual={}
for year in (2021,2022):
 pick=mask&(calendar.year==year)
 annual[str(year)]={"rows":int(pick.sum()),"Spearman":float(spearmanr(coordinate.loc[pick],labels.loc[pick,"joint_local_fee_net_advantage"]).statistic)}
assert annual["2021"]["rows"]>=70 and annual["2022"]["rows"]>=240
fixed=plan["necessary_gate"]["stationary_block_upper_below_zero"]
rng=np.random.default_rng(fixed["seed"]);n=len(calendar);p=1.0/fixed["mean_block_sessions"]
fx=coordinate.to_numpy(float);fy=labels.joint_local_fee_net_advantage.where(eligible).to_numpy(float)
rep=[]
for _ in range(fixed["replicates"]):
 idx=np.empty(n,dtype=int);idx[0]=rng.integers(n)
 for i in range(1,n):idx[i]=rng.integers(n) if rng.random()<p else (idx[i-1]+1)%n
 xx,yy=fx[idx],fy[idx];valid=np.isfinite(xx)&np.isfinite(yy)
 if valid.sum()>=100:
  value=float(spearmanr(xx[valid],yy[valid]).statistic)
  if np.isfinite(value):rep.append(value)
assert len(rep)>=fixed["minimum_valid_replicates"]
upper=float(np.quantile(rep,1-fixed["one_sided_alpha"]))
gates={"full_negative":full<0,"both_years_negative":all(z["Spearman"]<0 for z in annual.values()),"block_upper_below_zero":upper<0}
result={"status":"necessary_local_value_rank_pass" if all(gates.values()) else "not_promoted_fixed_pair_path_joint_value_rank_failed",
        "plan_sha256":sha(plan_path),"source_card":src["feature_card"],"label_recorder_id":src["label_recorder_id"],
        "full_calendar_rows":n,"source_known_rows":int(coordinate.notna().sum()),"mature_eligible_scored_rows":len(x),
        "full_Spearman":full,"annual":annual,"bootstrap":{"valid_replicates":len(rep),"mean_block_sessions":fixed["mean_block_sessions"],
        "one_sided_alpha":fixed["one_sided_alpha"],"upper_bound":upper},"gates":gates,
        "historical_first_public_qualified":False,"new_market_GETs":0,"model_fits":0,"native_accounts":0,"bootstrap_attempts":1,
        "SOXX_strategy_success":False,"independent_validation":False,
        "limits":["Rank correlation is a necessary local action-value screen, not incremental original39 prediction or a complete funded account",
                  "The 2021-2022 OOF training-reference evaluation is reused development history, not independent validation",
                  "Historical first receipt of minute bars is unqualified"]}
(card/"result.json").write_text(json.dumps(result,ensure_ascii=False,indent=2)+"\n")
print({"status":result["status"],"full":full,"annual":annual,"block_upper":upper,"gates":gates})


{'status': 'not_promoted_fixed_pair_path_joint_value_rank_failed', 'full': -0.11234095823954666, 'annual': {'2021': {'rows': 82, 'Spearman': -0.01646784799338782}, '2022': {'rows': 247, 'Spearman': -0.14124251043999272}}, 'block_upper': 0.061172164863221475, 'gates': {'full_negative': True, 'both_years_negative': True, 'block_upper_below_zero': False}}


## 中文结论

在已通过来源与重复性预检的固定因子上，按冻结假说检验“开盘两ETF分钟路径越分歧，09:46同状态增配SOXX／释放风险容量的0bp额外滑点、Tiered费用后局部净价值越低”。只读封存Qlib一分钟价格及原生局部价值标签；完整2021–2022 Qlib日历503日均保留，329日来源已知、可行动、标签成熟，2021为82日、2022为247日。无2023+市场结果读取、模型拟合或资金账户。

Spearman点值：全体 **-0.11234**、2021 **-0.01647**、2022 **-0.14124**，均符合预定负号。然而固定63会话平均区块、10000次重采样和单侧修正分位数下，相关系数的上界为 **+0.06117**，未低于0。三道必要门只过方向与逐年点值，统计稳定性失败；尤其2021几乎无排序支持。来源数值新颖不等于资金动作价值信息。

决定 `not_promoted_fixed_pair_path_joint_value_rank_failed`。该分钟数、平方能量公式、负号与09:46联合现金／上限局部标签的固定版本关闭；不翻符号、改时间、改分母、挑2022或接着做增量Ridge／完整账户来救援。历史分钟首次送达也未认证。此局部标签不是完整f696账户，复用2021–2022历史不是独立验证；SOXX费用后稳定超额、2/5bp与前向门均未证明。
